# Module 04 — Function Calling (Tool Use)

You give **`claude-opus-4-8`** a set of **tool definitions**. The model decides when to call them and returns a **`tool_use`** request; **you** execute the function locally and return a **`tool_result`**; the model then uses that result to produce the final answer.

This builds on **Modules 00–03** and uses the same **`AnthropicVertex` / ADC** path.

## Part B — Bootstrap

Setup mirrors Module 00 — see that module for full detail (prerequisites, ADC, logging).

In [ ]:
%pip install --quiet "anthropic[vertex]"

In [ ]:
# ===== CONDENSED BOOTSTRAP (mirrors Module 00) =====
import subprocess
from anthropic import AnthropicVertex

def _default_project() -> str:
    try:
        out = subprocess.run(
            ["gcloud", "config", "get-value", "project"],
            capture_output=True, text=True, timeout=15,
        ).stdout.strip()
        if out and out != "(unset)":
            return out
    except Exception:
        pass
    return "<YOUR_PROJECT_ID>"

PROJECT_ID = _default_project()   # or hard-code: PROJECT_ID = "my-project-id"
LOCATION   = "global"
MODEL      = "claude-opus-4-8"

assert PROJECT_ID and PROJECT_ID != "<YOUR_PROJECT_ID>", (
    "PROJECT_ID is still the placeholder. Run `gcloud config set project <id>` "
    "so it auto-detects, or edit PROJECT_ID directly."
)

client = AnthropicVertex(project_id=PROJECT_ID, region=LOCATION)
print(f"✅ Client ready (project={PROJECT_ID}, region={LOCATION}, model={MODEL}).")

## Part C — Define tools

A tool definition has three parts:

- **`name`** — the identifier the model calls.
- **`description`** — what the tool does and when to use it. The model relies on this; **good descriptions drive good tool use**.
- **`input_schema`** — a JSON Schema describing the expected arguments.

Below we define two **self-contained, local** tools (no external APIs or keys).

In [ ]:
# --- Tool definitions (the schema Claude sees) ---
tools = [
    {
        "name": "get_weather",
        "description": "Get the current weather for a location. Returns mock sample data for this tutorial.",
        "input_schema": {
            "type": "object",
            "properties": {
                "location": {"type": "string", "description": "City name, e.g. 'Paris'."},
            },
            "required": ["location"],
        },
    },
    {
        "name": "calculate",
        "description": "Perform a basic arithmetic operation on two numbers.",
        "input_schema": {
            "type": "object",
            "properties": {
                "operation": {
                    "type": "string",
                    "enum": ["add", "subtract", "multiply", "divide"],
                    "description": "The arithmetic operation to perform.",
                },
                "a": {"type": "number", "description": "The first operand."},
                "b": {"type": "number", "description": "The second operand."},
            },
            "required": ["operation", "a", "b"],
        },
    },
]

# --- Local implementations ---
def get_weather(location: str) -> str:
    # MOCK canned data — not a real weather service.
    return f"[MOCK] Weather in {location}: 18°C, partly cloudy, light wind."

def calculate(operation: str, a: float, b: float):
    # REAL local arithmetic.
    if operation == "add":
        return a + b
    if operation == "subtract":
        return a - b
    if operation == "multiply":
        return a * b
    if operation == "divide":
        return a / b if b != 0 else "error: division by zero"
    return f"error: unknown operation '{operation}'"

# Map tool name -> implementation.
TOOL_IMPLS = {"get_weather": get_weather, "calculate": calculate}
print("✅ Tools defined:", list(TOOL_IMPLS))

## Part D — One round, explicitly

When Claude wants a tool, the response's **`stop_reason`** is **`"tool_use"`** and its `content` contains a **`tool_use` block** with `.id`, `.name`, and `.input` (the arguments). Let's inspect one.

In [ ]:
resp = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    tools=tools,
    messages=[{"role": "user", "content": "What's the weather in Paris?"}],
)

print(f"stop_reason: {resp.stop_reason}")
for block in resp.content:
    if block.type == "tool_use":
        print(f"tool name : {block.name}")
        print(f"tool input: {block.input}")

## Part E — The complete tool-use loop (reusable)

The full pattern:

1. Send the conversation with `tools`.
2. If `stop_reason == "tool_use"`, **append the assistant turn** (its full `content`, which includes the `tool_use` blocks).
3. Run **every** requested tool and send back a **user turn** whose `content` is a list of **`tool_result`** blocks: `{"type": "tool_result", "tool_use_id": <id>, "content": <string>}`.
4. Repeat until `stop_reason` is no longer `"tool_use"`.

Claude may request **multiple tools in one turn** (parallel tool use), so handle all blocks each round.

In [ ]:
messages = [{"role": "user", "content": "What's the weather in Paris, and what is 23 times 19?"}]

max_rounds = 8   # guard: never loop forever
rounds = 0

while True:
    rounds += 1
    if rounds > max_rounds:
        print(f"⚠️  Hit max_rounds ({max_rounds}); breaking out of the tool-use loop.")
        break
    resp = client.messages.create(
        model=MODEL,
        max_tokens=1024,
        tools=tools,
        messages=messages,
    )
    if resp.stop_reason != "tool_use":
        break
    # Append the assistant's turn (includes the tool_use blocks).
    messages.append({"role": "assistant", "content": resp.content})
    # Execute every requested tool this round.
    results = []
    for block in resp.content:
        if block.type == "tool_use":
            out = TOOL_IMPLS[block.name](**block.input)
            print(f"  ↳ ran {block.name}({block.input}) -> {out}")
            results.append({
                "type": "tool_result",
                "tool_use_id": block.id,
                "content": str(out),
            })
    # Send the tool results back as a user turn.
    messages.append({"role": "user", "content": results})

# Final answer: the text block(s) from the last response.
final_text = "".join(b.text for b in resp.content if b.type == "text")
print("-" * 60)
print(final_text)

## Part F — Notes & recap

### Notes

- **`tool_choice`** controls whether/which tool is used: `"auto"` (default — model decides), `"any"` (must use some tool), a specific tool by name, or `"none"` (disable tools for this call).
- **Parallel tool calls:** one assistant turn can contain several `tool_use` blocks — always loop over all of them and return one `tool_result` per `tool_use_id`.
- **Description quality matters most:** clear tool/parameter descriptions are the biggest lever on whether the model calls the right tool with the right arguments.
- See the Anthropic **tool use** docs (https://docs.claude.com/en/docs/build-with-claude/tool-use) and confirm the current details there, as they may change.

### Recap

- You provide **tool definitions** (`name`, `description`, `input_schema`); the model returns **`tool_use`** requests.
- **You execute** the tool locally and return **`tool_result`** blocks keyed by `tool_use_id`.
- The **loop** continues while `stop_reason == "tool_use"` and handles **parallel** tool calls each round.
- `tool_choice` steers tool usage; **good descriptions** drive good tool use.

**Next: Module 05 — Extended thinking.**